# Extra experiments for the peer-review response

This notebook runs the new analyses and a longer CSDI training. It adds new files only; the original results and tables are not changed.


## Step 1. Connect Drive and check GPU

Expected output: `cuda: True` and a list that contains `extra_eval.py`, `csdi_long.py`. If it prints `cuda: False`, redo step 3 above (Runtime → Change runtime type → T4 GPU) and run this cell again. If it says the folder does not exist, fix `PROJECT_DIR` to the real path in Drive.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/mosd_project-main'
import os
assert os.path.isdir(PROJECT_DIR), 'Folder not found, fix PROJECT_DIR'
%cd {PROJECT_DIR}
import torch
print('cuda:', torch.cuda.is_available())
!ls tsi configs

Mounted at /content/drive
/content/drive/MyDrive/mosd_project-main
cuda: True
configs:
long.yaml  main.yaml  pilot.yaml

tsi:
chronos_imputer.py  data.py	      extra_eval.py	  kalman.py
common.py	    evaluate.py       extra_stats.py	  masks.py
csdi_imputer.py     extra_blend.py    extra_synthetic.py  metrics.py
csdi_long.py	    extra_chronos.py  __init__.py


## Step 2. Install packages

 If pip reports a version conflict and the cell fails, run the second line (without pinned versions) instead.

In [2]:
!pip -q install chronos-forecasting==2.3.2 pypots==1.5 tsdb==0.8 pyyaml
# fallback if the line above fails:
# !pip -q install chronos-forecasting pypots tsdb pyyaml

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.41.0 requires huggingface-hub<2.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


## Step 3. Download the data and check that it matches your original run

The first command downloads the three datasets again through `tsdb` (a few minutes). The second command recomputes all metrics from saved predictions and compares them with `results/tables/metrics_val.csv` and `metrics_test.csv`.


In [3]:
!python -m tsi.data --config configs/main.yaml
!python -m tsi.extra_eval --config configs/main.yaml --what check

2026-10-25 19:35:36 [WARNING]: ‼️ PyPOTS Ecosystem configuration file does not exist.
2026-10-25 19:35:36 [INFO]: Wrote new configs to config.ini successfully.
2026-10-25 19:35:36 [INFO]: 💫 Initialized PyPOTS Ecosystem configuration file /root/.pypots/config.ini successfully.
2026-10-25 19:35:37 [INFO]: You're using dataset electricity_transformer_temperature, please cite it properly in your work. You can find its reference information at the below link: 
https://github.com/WenjieDu/TSDB/tree/main/dataset_profiles/electricity_transformer_temperature
2026-10-25 19:35:37 [INFO]: Start downloading...
2026-10-25 19:35:38 [INFO]: Successfully downloaded data to /root/.pypots/tsdb/electricity_transformer_temperature/ETTm1.csv
2026-10-25 19:35:39 [INFO]: Successfully downloaded data to /root/.pypots/tsdb/electricity_transformer_temperature/ETTm2.csv
2026-10-25 19:35:39 [INFO]: Successfully downloaded data to /root/.pypots/tsdb/electricity_transformer_temperature/ETTh1.csv
2026-10-25 19:35:39 

## Step 4. Start the longer CSDI training (runs in the background)

This retrains CSDI on Electricity for 100 epochs with the learning rate dropping 10x at epochs 75 and 90 (the recipe of the original CSDI paper), then draws samples for validation and test. Training keeps running in parallel.

In [4]:
!mkdir -p logs
!nohup python -m tsi.csdi_long --config configs/long.yaml > logs/csdi_long.log 2>&1 &
import time; time.sleep(60)
!tail -n 5 logs/csdi_long.log

  return nn.TransformerEncoder(encoder_layer, num_layers=layers)
2026-10-25 19:37:45 [INFO]: CSDI initialized with the given hyperparameters, the number of trainable parameters: 413,521
2026-10-25 19:37:59 [INFO]: Epoch 001 - training loss (default): 0.4370, validation loss: 0.4040
2026-10-25 19:38:09 [INFO]: Epoch 002 - training loss (default): 0.3983, validation loss: 0.4005
2026-10-25 19:38:20 [INFO]: Epoch 003 - training loss (default): 0.3782, validation loss: 0.4023


## Scheduler fix and training retry

In [14]:
%%writefile tsi/csdi_long.py
import math
import time
import zlib

import numpy as np
import torch
from pypots.imputation import CSDI
from pypots.optim import Adam
from pypots.optim.lr_scheduler import MultiStepLR

from .common import (load_config, load_masks, load_series, mask_configs, masked_windows, parse_args,
                     pred_path, results_dir, save_json)
from .csdi_imputer import draw_samples, training_windows


def build(cfg, name, lcfg):
    c = cfg['csdi']
    steps = math.ceil(c['n_train_windows'] / c['batch_size']) if lcfg.get('scheduler_unit', 'step') == 'step' else 1
    milestones = [int(f * lcfg['epochs'] * steps) for f in lcfg['milestones']]
    print(f'scheduler unit: {lcfg.get("scheduler_unit", "step")}, steps per epoch: {steps}, milestones: {milestones}')
    optimizer = Adam(lr=lcfg['lr'], weight_decay=lcfg['weight_decay'],
                     lr_scheduler=MultiStepLR(milestones=milestones, gamma=lcfg['gamma']))
    ckpt = results_dir(cfg) / 'csdi_long' / name / 'model.pypots'
    torch.manual_seed(lcfg['seed'])
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(lcfg['seed'])
    model = CSDI(n_steps=cfg['window'], n_features=1, n_layers=c['n_layers'], n_heads=c['n_heads'],
                 n_channels=c['n_channels'], d_time_embedding=c['d_time_embedding'],
                 d_feature_embedding=c['d_feature_embedding'], d_diffusion_embedding=c['d_diffusion_embedding'],
                 n_diffusion_steps=c['n_diffusion_steps'], target_strategy='mix', batch_size=c['batch_size'],
                 epochs=lcfg['epochs'], patience=lcfg['patience'], optimizer=optimizer, device=c['device'],
                 saving_path=str(ckpt.parent / 'runs'), model_saving_strategy='best', verbose=True)
    return model, ckpt, milestones


def final_lr(model):
    try:
        return model.optimizer.torch_optimizer.param_groups[0]['lr']
    except Exception:
        return None


def main():
    def extra(p):
        p.add_argument('--skip-train', action='store_true')
    args = parse_args('CSDI with a longer schedule and step-wise learning rate decay (original CSDI recipe)', extra)
    cfg = load_config(args.config)
    lcfg, c = cfg['csdi_long'], cfg['csdi']
    L, edge, max_hidden = cfg['window'], cfg['edge'], max(cfg['hidden_counts'])
    qs = np.array(cfg['quantiles'])
    log = {}
    for name in lcfg['datasets']:
        series = load_series(cfg, name)
        model, ckpt, milestones = build(cfg, name, lcfg)
        if args.skip_train:
            model.load(str(ckpt))
        else:
            rng = np.random.default_rng([cfg['seeds']['train'], zlib.crc32(name.encode())])
            tr = training_windows(series, 'train', c['n_train_windows'], L, edge, max_hidden, rng)
            va = training_windows(series, 'val', c['n_val_windows'], L, edge, max_hidden, rng)
            va_x = np.where(va['mask'] == 1, va['X'], np.nan)
            t0 = time.time()
            model.fit({'X': tr['X'], 'for_pattern_mask': tr['mask']}, {'X': va_x, 'X_ori': va['X']})
            hours = (time.time() - t0) / 3600
            ckpt.parent.mkdir(parents=True, exist_ok=True)
            model.save(str(ckpt), overwrite=True)
            log[name] = {'train_hours': hours, 'best_val_loss': float(model.best_loss), 'epochs': lcfg['epochs'],
                         'milestones': milestones, 'final_lr': final_lr(model)}
            print(f'{name}: trained in {hours:.2f} h, final learning rate {final_lr(model)} (expected {lcfg["lr"] * lcfg["gamma"] ** 2:.0e})')
        torch.manual_seed(lcfg['seed'] + 1)
        for split in ('val', 'test'):
            masks = load_masks(cfg, split, name)
            out = {}
            t0 = time.time()
            for key in mask_configs(cfg):
                x, idx = masked_windows(series, masks, key)
                samples = draw_samples(model, x, c['n_samples'], c['sample_batch_size'])
                samples = np.take_along_axis(samples, idx[:, None, :], axis=2)
                out[f'{key}/samples'] = samples.astype(np.float32)
                out[f'{key}/quantiles'] = np.moveaxis(np.quantile(samples, qs, axis=1), 0, -1)
            np.savez(pred_path(cfg, split, name, 'csdi_long'), **out)
            log.setdefault(name, {})[f'{split}_sampling_hours'] = (time.time() - t0) / 3600
            print(f'{split}/{name}: sampling done in {(time.time() - t0) / 3600:.2f} h')
    if log:
        save_json(log, results_dir(cfg) / 'csdi_long' / ('log_skip_train.json' if args.skip_train else 'log.json'))


if __name__ == '__main__':
    main()

Overwriting tsi/csdi_long.py


In [15]:
!pkill -f tsi.csdi_long
!mv logs/csdi_long.log logs/failed_scheduler_run.txt
!nohup python -m tsi.csdi_long --config configs/long.yaml > logs/csdi_long.log 2>&1 &
import time; time.sleep(90)
!grep -E "scheduler unit|Epoch" logs/csdi_long.log | head -n 12

2026-10-25 21:01:06 [INFO]: Epoch 001 - training loss (default): 0.4319, validation loss: 0.3881
2026-10-25 21:01:16 [INFO]: Epoch 002 - training loss (default): 0.3625, validation loss: 0.3786
2026-10-25 21:01:26 [INFO]: Epoch 003 - training loss (default): 0.3513, validation loss: 0.3732
2026-10-25 21:01:37 [INFO]: Epoch 004 - training loss (default): 0.3490, validation loss: 0.3750
2026-10-25 21:01:47 [INFO]: Epoch 005 - training loss (default): 0.3580, validation loss: 0.3744
2026-10-25 21:01:58 [INFO]: Epoch 006 - training loss (default): 0.3569, validation loss: 0.3680
2026-10-25 21:02:09 [INFO]: Epoch 007 - training loss (default): 0.3546, validation loss: 0.3751


In [17]:
!grep -n "steps per epoch" tsi/csdi_long.py

20:    print(f'scheduler unit: {lcfg.get("scheduler_unit", "step")}, steps per epoch: {steps}, milestones: {milestones}')


## Step 5. Analyses that do not need CSDI

* **5a** compares three ways to combine the left and right Chronos forecasts (about 1 minute).
* **5b** reruns Chronos with 21 quantile levels and with matched context lengths (a few minutes). Expected last line of the first part: `max abs difference to the saved 11-level quantiles ...` with a value of about `1e-3` or smaller. A much larger value means the Chronos version differs from original run.
* **5c** runs the synthetic tests (oracle for the blend, time reversal).

In [5]:
!python -m tsi.extra_blend --config configs/main.yaml

val: 60 rows -> /content/drive/MyDrive/mosd_project-main/results/tables/extra_blend_variants_val.csv
test: 60 rows -> /content/drive/MyDrive/mosd_project-main/results/tables/extra_blend_variants_test.csv


In [6]:
!python -m tsi.extra_chronos --config configs/main.yaml --what q21 matched

I0000 00:00:1792957142.516993    2990 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792957146.138725    2990 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
config.json: 100% 969/969 [00:00<00:00, 6.36MB/s]
model.safetensors: 100% 112M/112M [00:01<00:00, 72.0MB/s]
val/ETT_h1: q21 done in 137.4s
val/Electricity: q21 done in 139.9s
val/BeijingAir: q21 done in 139.2s
test/ETT_h1: q21 done in 144.2s
test/Electricity: q21 done in 139.6s
test/BeijingAir: q21 done in 139.8s
max abs difference to the saved 11-level quantiles (scaled units): 2.98e-04
val/ETT_h1: matched context done in 2.5s
val/Electricity: matched context done in 2.5s
val/BeijingAir: matched context done in 2.5s
test/ETT_h1: matched context done in 2.5s
test/Ele

In [7]:
!python -m tsi.extra_synthetic --config configs/main.yaml --n-windows 200

I0000 00:00:1792958629.938453    9198 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792958633.200605    9198 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
oracle test: ratio 0.01, gap 24 done
oracle test: ratio 0.01, gap 128 done
oracle test: ratio 0.1, gap 24 done
oracle test: ratio 0.1, gap 128 done
oracle test: ratio 1.0, gap 24 done
oracle test: ratio 1.0, gap 128 done
 noise_ratio  gap   model    MAE  pinball   CRPS    PCE     CCE  cov80  width80    SIW  width_ratio_to_oracle
        0.01   24  oracle 1.6397   0.6330 1.1116 0.0251  0.0016 0.7946   5.2095 0.1925                  1.000
        0.01   24    left 2.8183   1.1028 1.9460 0.0306  0.0672 0.7238   7.9286 0.2855                  1.522
        0.01   24   ri

In [8]:
import pandas as pd; pd.read_csv('results/tables/extra_synthetic_oracle.csv').query('gap==128')

,noise_ratio,gap,model,MAE,pinball,CRPS,PCE,CCE,cov80,width80,SIW,width_ratio_to_oracle
6,0.01,128,oracle,3.6506,1.4160,2.4893,0.0173,0.0070,0.7882,11.5186,0.3925,1.000
7,0.01,128,left,6.4893,2.6061,4.6546,0.0564,0.1225,0.6444,15.9707,0.5336,1.387
8,0.01,128,right,6.7455,2.7062,4.8359,0.0699,0.1358,0.6246,15.6095,0.5192,1.355
9,0.01,128,linear,3.9734,1.5564,2.7536,0.0170,0.0190,0.7781,13.0967,0.4366,1.137
10,0.01,128,uniform,5.0766,1.9923,3.5315,0.0243,0.0411,0.7490,15.7901,0.5264,1.371
11,0.01,128,poe,4.5196,1.8417,3.3193,0.0890,0.1923,0.5410,8.5807,0.2924,0.745
18,0.10,128,oracle,3.6917,1.4214,2.4949,0.0167,0.0118,0.7875,11.5672,0.4230,1.000
19,0.10,128,left,6.4688,2.6237,4.6991,0.0584,0.1271,0.6324,15.3484,0.5455,1.327
20,0.10,128,right,6.6345,2.6466,4.7297,0.0570,0.1212,0.6446,16.3135,0.5835,1.410
21,0.10,128,linear,4.0681,1.5854,2.8011,0.0160,0.0310,0.7670,13.0522,0.4652,1.128


In [9]:
!python -m tsi.extra_synthetic --config configs/main.yaml --n-windows 1000

I0000 00:00:1792958938.350211   10550 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792958941.670021   10550 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
oracle test: ratio 0.01, gap 24 done
oracle test: ratio 0.01, gap 128 done
oracle test: ratio 0.1, gap 24 done
oracle test: ratio 0.1, gap 128 done
oracle test: ratio 1.0, gap 24 done
oracle test: ratio 1.0, gap 128 done
 noise_ratio  gap   model    MAE  pinball   CRPS    PCE     CCE  cov80  width80    SIW  width_ratio_to_oracle
        0.01   24  oracle 1.6210   0.6273 1.1031 0.0132 -0.0014 0.8011   5.2095 0.1952                  1.000
        0.01   24    left 2.8923   1.1434 2.0241 0.0382  0.0819 0.6977   7.8159 0.2844                  1.500
        0.01   24   ri

## Step 6. Tables and figures from the saved predictions

Computes: CRPS on one scale for all models, per-channel calibration, paired bootstrap with confidence intervals, matched left/right comparison, and CSDI validation curves. Last lines of the output print counts such as `winner changes between original CRPS and CRPS_q21 in N of 12 cells` and `pinball comparisons: first better=..., second better=..., tie=...`.

In [32]:
!python -m tsi.extra_eval --config configs/main.yaml --what crps channels bootstrap matched curves

val: winner changes between original CRPS and CRPS_q21 in 0 of 12 cells
test: winner changes between original CRPS and CRPS_q21 in 2 of 12 cells
val: per-channel tables and figure written
test: per-channel tables and figure written
val: pinball comparisons: first better=42, tie=16, second better=6
test: pinball comparisons: first better=44, tie=15, second better=5
val: matched left/right table written
test: matched left/right table written
             run  epochs  best_epoch  best_val_loss  last_val_loss
Electricity_long     100          92         0.1877         0.1893
      BeijingAir      35          30         0.1454         0.1467
          ETT_h1      31          26         0.1966         0.1981
     Electricity      40          39         0.2117         0.2154


## Step 7. Wait for CSDI to finish

Training is finished when the log contains the lines `val/Electricity: sampling done` and `test/Electricity: sampling done`, and the training rate is equal to `1e-05`.


In [29]:
!tail -n 6 logs/csdi_long.log
!nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv

ai4ts v0.0.3 - building AI for unified time-series analysis, https://time-series.ai 

scheduler unit: step, steps per epoch: 188, milestones: [14100, 16920]
Electricity: trained in 0.30 h, final learning rate 1e-05 (expected 1e-05)
val/Electricity: sampling done in 0.10 h
test/Electricity: sampling done in 0.10 h
utilization.gpu [%], memory.used [MiB]
0 %, 3 MiB


In [24]:
!grep -E "final learning rate|trained in|sampling done" logs/csdi_long.log

## Step 8. Tables with the 100-epoch CSDI

Cell adds the `csdi_long` model to the comparison tables and the validation curves.

In [30]:
!python -m tsi.extra_eval --config configs/main.yaml --what crps bootstrap curves

val: winner changes between original CRPS and CRPS_q21 in 0 of 12 cells
test: winner changes between original CRPS and CRPS_q21 in 2 of 12 cells
val: pinball comparisons: first better=42, tie=16, second better=6
test: pinball comparisons: first better=44, tie=15, second better=5
             run  epochs  best_epoch  best_val_loss  last_val_loss
Electricity_long     100          92         0.1877         0.1893
      BeijingAir      35          30         0.1454         0.1467
          ETT_h1      31          26         0.1966         0.1981
     Electricity      40          39         0.2117         0.2154


In [33]:
p = 'tsi/extra_eval.py'
s = open(p).read()
if "('csdi_long', 'kalman')" not in s:
    s = s.replace("('csdi_long', 'csdi')]", "('csdi_long', 'csdi'),\n         ('csdi_long', 'chronos_blend'), ('csdi_long', 'kalman')]")
    s = s.replace("diff_rows, cov_rows, win_rows = [], [], []", "diff_rows, cov_rows, win_rows, long_rows = [], [], [], []")
    a = "                win_rows.append({**base, 'best_pinball': MAIN[best], 'tied_with_best': ','.join(tied) or '-'})\n"
    s = s.replace(a, a + "            ML = MAIN + ['csdi_long']\n            if all((m, 'pinball') in pos for m in ML):\n                ip = [pos[(m, 'pinball')] for m in ML]\n                best = int(np.argmin(point[ip]))\n                tied = [ML[i] for i in range(len(ML)) if i != best and\n                        verdict(*interval(B[:, ip[i]] - B[:, ip[best]])) == 'tie']\n                long_rows.append({**base, 'best_pinball': ML[best], 'tied_with_best': ','.join(tied) or '-'})\n")
    b = "        pd.DataFrame(win_rows).to_csv(tables_dir(cfg) / f'extra_winners_with_ties_{split}.csv', index=False)\n"
    s = s.replace(b, b + "        pd.DataFrame(long_rows).to_csv(tables_dir(cfg) / f'extra_winners_with_ties_long_{split}.csv', index=False)\n")
    open(p, 'w').write(s)
print('patched' if "('csdi_long', 'kalman')" in open(p).read() else 'FAILED')
!python -m tsi.extra_eval --config configs/main.yaml --what bootstrap
!zip -rq extra_tables.zip results/tables/extra_*
from google.colab import files
files.download('extra_tables.zip')

patched
val: pinball comparisons: first better=49, tie=17, second better=6
test: pinball comparisons: first better=50, tie=17, second better=5


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Looking for what went wrong with CSDI training.

For some reason, initial training did not work well. Need to investigate.

Edit: found the problem - The schedule was meant to follow the original CSDI recipe: learning rate $10^{-3}$, divided by 10 at 75% and again at 90% of training. We set the milestones in epochs (`[75, 90]`), but PyPOTS calls `scheduler.step()` after every batch, not after every epoch. With 188 batches per epoch, both drops happened within the first half epoch, and the model trained at

$$\eta = 10^{-3} \cdot 0.1^2 = 10^{-5}$$

for the remaining 99.5 epochs. Validation loss stayed at about 0.37 and never left the plateau that the original 40-epoch run also shows for the first ~11 epochs. The log of this invalid run is kept as `logs/failed_scheduler_run.txt`.

**Fix.** The milestones are now counted in optimizer steps:

$$0.75 \cdot 100 \cdot 188 = 14100, \qquad 0.90 \cdot 100 \cdot 188 = 16920.$$

The run prints `scheduler unit: step, steps per epoch: 188, milestones: [14100, 16920]` at the start and `final learning rate 1e-05` at the end, which confirms the schedule worked.

**Result of the corrected run.** Validation loss leaves the plateau around epoch 15 and keeps falling after each learning-rate drop (about 0.19 at epoch 88, against 0.2117 for the original 40-epoch model). This is the run we use to answer the reviewers' question about whether CSDI was trained long enough.


In [12]:
!grep -E "Epoch (001|010|020|040|060|075|076|080|090|091|100) " logs/csdi_long.log
!grep -n "final learning rate" logs/csdi_long.log

2026-10-25 19:37:59 [INFO]: Epoch 001 - training loss (default): 0.4370, validation loss: 0.4040
2026-10-25 19:39:46 [INFO]: Epoch 010 - training loss (default): 0.3773, validation loss: 0.3928
2026-10-25 19:42:04 [INFO]: Epoch 020 - training loss (default): 0.3703, validation loss: 0.3853
2026-10-25 19:46:42 [INFO]: Epoch 040 - training loss (default): 0.3514, validation loss: 0.3776
2026-10-25 19:51:19 [INFO]: Epoch 060 - training loss (default): 0.3523, validation loss: 0.3739
2026-10-25 19:54:30 [INFO]: Epoch 075 - training loss (default): 0.3449, validation loss: 0.3718
2026-10-25 19:54:41 [INFO]: Epoch 076 - training loss (default): 0.3495, validation loss: 0.3710
2026-10-25 19:55:25 [INFO]: Epoch 080 - training loss (default): 0.3447, validation loss: 0.3703
2026-10-25 19:57:14 [INFO]: Epoch 090 - training loss (default): 0.3428, validation loss: 0.3707
2026-10-25 19:57:25 [INFO]: Epoch 091 - training loss (default): 0.3455, validation loss: 0.3712
2026-10-25 19:59:04 [INFO]: Ep

In [13]:
import pypots, os
!grep -rn "lr_scheduler" {os.path.dirname(pypots.__file__)} | grep -v pyc | head -40


████████╗██╗███╗   ███╗███████╗    ███████╗███████╗██████╗ ██╗███████╗███████╗    █████╗ ██╗
╚══██╔══╝██║████╗ ████║██╔════╝    ██╔════╝██╔════╝██╔══██╗██║██╔════╝██╔════╝   ██╔══██╗██║
   ██║   ██║██╔████╔██║█████╗█████╗███████╗█████╗  ██████╔╝██║█████╗  ███████╗   ███████║██║
   ██║   ██║██║╚██╔╝██║██╔══╝╚════╝╚════██║██╔══╝  ██╔══██╗██║██╔══╝  ╚════██║   ██╔══██║██║
   ██║   ██║██║ ╚═╝ ██║███████╗    ███████║███████╗██║  ██║██║███████╗███████║██╗██║  ██║██║
   ╚═╝   ╚═╝╚═╝     ╚═╝╚══════╝    ╚══════╝╚══════╝╚═╝  ╚═╝╚═╝╚══════╝╚══════╝╚═╝╚═╝  ╚═╝╚═╝
ai4ts v0.0.3 - building AI for unified time-series analysis, https://time-series.ai 

grep: /usr/local/lib/python3.13/dist-packages/pypots/optim/__pycache__/sgd.cpython-313.pyc: binary file matches
grep: /usr/local/lib/python3.13/dist-packages/pypots/optim/__pycache__/adadelta.cpython-313.pyc: binary file matches
grep: /usr/local/lib/python3.13/dist-packages/pypots/optim/__pycache__/adamw.cpython-313.pyc: binary file matches
grep: /usr/l

## Step 9. Download the results

The cell below packs everything new into `extra_results.zip` and starts the download in browser. Same files are copied to the Drive folder under `results/`.


In [31]:
!zip -rq extra_results.zip results/tables/extra_* results/figures/extra_* results/csdi_long logs/csdi_long.log results/predictions/*/*/csdi_long.npz results/predictions/*/*/chronos_*_q21.npz results/predictions/*/*/chronos_leftm.npz results/predictions/*/*/chronos_rightm.npz
from google.colab import files
files.download('extra_results.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Step 10. Additional training (for ETT_h1 and BeijingAir datasets)

Log copy of electricity.json file

In [34]:
!cp results/csdi_long/log.json results/csdi_long/log_electricity.json
!ls results/csdi_long

Electricity  log_electricity.json  log.json


Config

In [35]:
%%writefile configs/long2.yaml
base: main.yaml
csdi:
  device: cuda
  sample_batch_size: 400
csdi_long:
  datasets: [ETT_h1, BeijingAir]
  epochs: 100
  patience: 100
  lr: 0.001
  weight_decay: 0.000001
  milestones: [0.75, 0.9]
  gamma: 0.1
  seed: 0

Writing configs/long2.yaml


Launch

In [36]:
!nohup python -m tsi.csdi_long --config configs/long2.yaml > logs/csdi_long_2.log 2>&1 &
import time; time.sleep(90)
!grep -E "Epoch" logs/csdi_long_2.log | tail -n 3

2026-10-25 22:30:51 [INFO]: Epoch 005 - training loss (default): 0.3066, validation loss: 0.2841
2026-10-25 22:31:02 [INFO]: Epoch 006 - training loss (default): 0.3053, validation loss: 0.2809
2026-10-25 22:31:13 [INFO]: Epoch 007 - training loss (default): 0.2887, validation loss: 0.2655


Monitoring

In [51]:
!pgrep -f tsi.csdi_long || echo finished
!grep -E "Epoch" logs/csdi_long_2.log | tail -n 3
!grep -E "scheduler unit|trained in|sampling done" logs/csdi_long_2.log

62356
2026-10-25 23:17:37 [INFO]: Epoch 098 - training loss (default): 0.1499, validation loss: 0.1341
2026-10-25 23:17:48 [INFO]: Epoch 099 - training loss (default): 0.1525, validation loss: 0.1346
2026-10-25 23:17:59 [INFO]: Epoch 100 - training loss (default): 0.1486, validation loss: 0.1338
scheduler unit: step, steps per epoch: 188, milestones: [14100, 16920]
ETT_h1: trained in 0.30 h, final learning rate 1e-05 (expected 1e-05)
val/ETT_h1: sampling done in 0.10 h
test/ETT_h1: sampling done in 0.10 h
scheduler unit: step, steps per epoch: 188, milestones: [14100, 16920]
BeijingAir: trained in 0.30 h, final learning rate 1e-05 (expected 1e-05)
val/BeijingAir: sampling done in 0.10 h
test/BeijingAir: sampling done in 0.10 h


Collecting tables again

In [52]:
!python -m tsi.extra_eval --config configs/main.yaml --what crps bootstrap curves

val: winner changes between original CRPS and CRPS_q21 in 0 of 12 cells
test: winner changes between original CRPS and CRPS_q21 in 2 of 12 cells
val: pinball comparisons: first better=67, tie=23, second better=6
test: pinball comparisons: first better=70, tie=21, second better=5
             run  epochs  best_epoch  best_val_loss  last_val_loss
Electricity_long     100          92         0.1877         0.1893
     ETT_h1_long     100          97         0.1702         0.1705
 BeijingAir_long     100          94         0.1334         0.1338
      BeijingAir      35          30         0.1454         0.1467
          ETT_h1      31          26         0.1966         0.1981
     Electricity      40          39         0.2117         0.2154


Download

In [53]:
!zip -rq extra_tables_2.zip results/tables/extra_* results/figures/extra_* results/csdi_long/log*.json logs/csdi_long*.log
from google.colab import files
files.download('extra_tables_2.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>